In [1]:
import numpy as np
import pandas as pd

In [2]:
TOXIC_TOTAL = pd.read_csv('data/TOXIC_TOTAL_labeled.csv')
TOXIC_TOTAL['label_str'] = TOXIC_TOTAL['label'].astype(str)
# TOXIC_TOTAL['label'] = TOXIC_TOTAL['label'].astype(str)
# TOXIC_TOTAL.to_csv('data/TOXIC_TOTAL_labeled.csv', index=False)

In [3]:
# TOXIC_TOTAL['label'].value_counts()
TOXIC_TOTAL['origin'].value_counts()

DILI    475
OFT      62
T3DB     20
US       16
Name: origin, dtype: int64

In [4]:
# TOXIC_TOTAL['label_str'].value_counts()
type(TOXIC_TOTAL['label'].values[0])

numpy.float64

In [7]:
pubchemmerge_clean = pd.read_csv('data/TOXIC_TOTAL_labeled.csv')
pubchemmerge_clean = pubchemmerge_clean[(pubchemmerge_clean['label']==1) & (pubchemmerge_clean['origin'] == "DILI")]
pubchemmerge_clean

,Drug_ID,Drug,Y,origin,SMILES,Substance_Name,Toxicity_Flag,canonical_smiles,DB_Chemical_name,Hepatotoxicity,Number,ID,label
11,938.0,O=C(O)c1cccnc1,1.0,DILI,O=C(O)c1cccnc1,NaN,NaN,NaN,NaN,NaN,NaN,11_DILI,1.0
14,1046.0,NC(=O)c1cnccn1,1.0,DILI,NC(=O)c1cnccn1,NaN,NaN,NaN,NaN,NaN,NaN,14_DILI,1.0
17,1674.0,Cc1ccc2cc3c(ccc4ccccc43)c3c2c1CC3,1.0,DILI,Cc1ccc2cc3c(ccc4ccccc43)c3c2c1CC3,NaN,NaN,NaN,NaN,NaN,NaN,17_DILI,1.0
18,1775.0,O=C1NC(=O)C(c2ccccc2)(c2ccccc2)N1,1.0,DILI,O=C1NC(=O)C(c2ccccc2)(c2ccccc2)N1,NaN,NaN,NaN,NaN,NaN,NaN,18_DILI,1.0
19,1923.0,Oc1cccc2cccnc12,1.0,DILI,Oc1cccc2cccnc12,NaN,NaN,NaN,NaN,NaN,NaN,19_DILI,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
465,9915926.0,Cc1ccc(-n2[nH]c(C)c(NN=C3C=CC=C(c4cccc(C(=O)O)...,1.0,DILI,Cc1ccc(-n2[nH]c(C)c(NN=C3C=CC=C(c4cccc(C(=O)O)...,NaN,NaN,NaN,NaN,NaN,NaN,465_DILI,1.0
466,10113978.0,Cc1ccc(Nc2nccc(N(C)c3ccc4c(C)n(C)nc4c3)n2)cc1S...,1.0,DILI,Cc1ccc(Nc2nccc(N(C)c3ccc4c(C)n(C)nc4c3)n2)cc1S...,NaN,NaN,NaN,NaN,NaN,NaN,466_DILI,1.0
470,23663956.0,CCCC(CCC)C(=O)O.CCCC(CCC)C(=O)[O-].[Na+],1.0,DILI,CCCC(CCC)C(=O)O.CCCC(CCC)C(=O)[O-].[Na+],NaN,NaN,NaN,NaN,NaN,NaN,470_DILI,1.0
472,54675785.0,C=C1c2cccc(O)c2C(O)=C2C(=O)C3(O)C(O)=C(C(N)=O)...,1.0,DILI,C=C1c2cccc(O)c2C(O)=C2C(=O)C3(O)C(O)=C(C(N)=O)...,NaN,NaN,NaN,NaN,NaN,NaN,472_DILI,1.0


In [9]:
# murcho scaffold split
pubchemmerge_clean = pd.read_csv('data/TOXIC_TOTAL_labeled.csv')
pubchemmerge_clean = pubchemmerge_clean[(pubchemmerge_clean['label']==1) & (pubchemmerge_clean['origin'] == "DILI")] # 236개

from collections import defaultdict

import numpy
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
import os

from chainer_chemistry.dataset.splitters.base_splitter import BaseSplitter


def generate_scaffold(smiles, include_chirality=False):
    """return scaffold string of target molecule"""
    mol = Chem.MolFromSmiles(smiles)
    scaffold = MurckoScaffold\
        .MurckoScaffoldSmiles(mol=mol, includeChirality=include_chirality)
    return scaffold


class ScaffoldSplitter(BaseSplitter):
    """Class for doing data splits by chemical scaffold.

    Referred Deepchem for the implementation, https://git.io/fXzF4
    """
    def _split(self, dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1,
               **kwargs):
        numpy.testing.assert_almost_equal(frac_train + frac_valid + frac_test,
                                          1.)
        seed = kwargs.get('seed', None)
        smiles_list = kwargs.get('smiles_list')
        include_chirality = kwargs.get('include_chirality')
        if len(dataset) != len(smiles_list):
            raise ValueError("The lengths of dataset and smiles_list are "
                             "different")

        rng = numpy.random.RandomState(seed)

        scaffolds = defaultdict(list) #! scaffold-compound dictionary
        for ind, smiles in enumerate(smiles_list):
            scaffold = generate_scaffold(smiles, include_chirality)
            scaffolds[scaffold].append(ind)

        # scaffold_sets = rng.permutation(list(scaffolds.values()))

        # n_total_valid = int(numpy.floor(frac_valid * len(dataset)))
        # n_total_test = int(numpy.floor(frac_test * len(dataset)))
        
        scaffold_sets = list(scaffolds.values())
        rng.shuffle(scaffold_sets)

        train_index = []
        valid_index = []
        test_index = []

        for scaffold_set in scaffold_sets:
            if len(valid_index) < frac_valid * len(dataset):
            # if len(valid_index) + len(scaffold_set) <= n_total_valid:
                valid_index.extend(scaffold_set)
            elif len(test_index) < frac_test * len(dataset):
            # elif len(test_index) + len(scaffold_set) <= n_total_test:
                test_index.extend(scaffold_set)
            else:
                train_index.extend(scaffold_set)

        return numpy.array(train_index), numpy.array(valid_index),\
            numpy.array(test_index),\


    def train_valid_test_split(self, dataset, smiles_list, frac_train=0.8,
                               frac_valid=0.1, frac_test=0.1, converter=None,
                               return_index=True, seed=None,
                               include_chirality=False, **kwargs):
        """Split dataset into train, valid and test set.

        Split indices are generated by splitting based on the scaffold of small
        molecules.

        Args:
            dataset(NumpyTupleDataset, numpy.ndarray):
                Dataset.
            smiles_list(list):
                SMILES list corresponding to datset.
            seed (int):
                Random seed.
            frac_train(float):
                Fraction of dataset put into training data.
            frac_valid(float):
                Fraction of dataset put into validation data.
            converter(callable):
            return_index(bool):
                If `True`, this function returns only indices. If `False`, this
                function returns splitted dataset.

        Returns:
            SplittedDataset(tuple): splitted dataset or indices

        """
        return super(ScaffoldSplitter, self)\
            .train_valid_test_split(dataset, frac_train, frac_valid, frac_test,
                                    converter, return_index, seed=seed,
                                    smiles_list=smiles_list,
                                    include_chirality=include_chirality,
                                    **kwargs)

    def train_valid_split(self, dataset, smiles_list, frac_train=0.9,
                          frac_valid=0.1, converter=None, return_index=True,
                          seed=None, include_chirality=False, **kwargs):
        """Split dataset into train and valid set.

        Split indices are generated by splitting based on the scaffold of small
        molecules.

        Args:
            dataset(NumpyTupleDataset, numpy.ndarray):
                Dataset.
            smiles_list(list):
                SMILES list corresponding to datset.
            seed (int):
                Random seed.
            frac_train(float):
                Fraction of dataset put into training data.
            frac_valid(float):
                Fraction of dataset put into validation data.
            converter(callable):
            return_index(bool):
                If `True`, this function returns only indices. If `False`, this
                function returns splitted dataset.

        Returns:
            SplittedDataset(tuple): splitted dataset or indices

        """
        return super(ScaffoldSplitter, self)\
            .train_valid_split(dataset, frac_train, frac_valid, converter,
                               return_index, seed=seed,
                               smiles_list=smiles_list,
                               include_chirality=include_chirality, **kwargs)

# from chainer_chemistry.dataset.splitters.scaffold_splitter import ScaffoldSplitter

# smiles_list = pubchemmerge_clean['CAN_SMILES'].tolist()
smiles_list = pubchemmerge_clean['SMILES'].tolist()
dataset = np.arange(len(pubchemmerge_clean))

splitter = ScaffoldSplitter()

# Perform the split with frac_valid set to 0
train_indices, valid_indices, test_indices = splitter.train_valid_test_split(
    pubchemmerge_clean,
    smiles_list= smiles_list,
    frac_train=0.8,
    frac_valid=0.0,
    frac_test=0.2,
    seed=42
)

# Print the split indices
print("Train indices:", train_indices)
print("Valid indices:", valid_indices)
print("Test indices:", test_indices)


len(train_indices)
len(test_indices)
0 in train_indices
pubchemmerge_clean_sample_test = pubchemmerge_clean.loc[test_indices]
pubchemmerge_clean_sample_test['label'].value_counts()
import numpy as np
train_indices.max()
# valid_indices.max()
test_indices.max()

# Check for overlapping indices
overlap_indices = np.intersect1d(train_indices, np.intersect1d(valid_indices, test_indices))

if len(overlap_indices) > 0:
    print("There are overlapping indices between train_indices, valid_indices, and test_indices.")
    print("Overlapping indices:", overlap_indices)
else:
    print("There are no overlapping indices between train_indices, valid_indices, and test_indices.")

import pandas as pd

# pubchemmerge_clean 데이터프레임을 사용하는 가정
# train_indices, valid_indices, test_indices를 이미 구했음

# 인덱스를 사용해 train 및 test 데이터프레임을 생성
train_df = pubchemmerge_clean.iloc[train_indices]
test_df = pubchemmerge_clean.iloc[test_indices]

# train_df
# # CSV 파일로 저장
os.makedirs('./source_SUBDYVE/SSM/data/TOXIC/', exist_ok=True)
# train_df.to_csv('./source_SUBDYVE/SSM/data/TOXIC/train_data.csv', index=False)
# test_df.to_csv('./source_SUBDYVE/SSM/data/TOXIC/test_data.csv', index=False)

# test_df = test_df[['SMILES', 'label', 'CID', 'CompoundName']]
train_df.reset_index(drop=True, inplace=True)
train_df.index.name = 'index'
train_df.to_csv('./source_SUBDYVE/SSM/data/TOXIC/train_data.csv')
test_df.reset_index(drop=True, inplace=True)
test_df.index.name = 'index'
test_df.to_csv('./source_SUBDYVE/SSM/data/TOXIC/test_data.csv')


ModuleNotFoundError: No module named 'distutils.msvccompiler'

In [1]:
# DILI = pd.read_csv('data/DILI.csv')
# OFT = pd.read_csv('data/OpenFoodTox_non_toxic_salt_filtered.csv')
# T3DB = pd.read_csv('data/T3DB_food_hepototixic_salt_filtered.csv')
# US = pd.read_csv('data/useful_substances.csv')

# # 각 DataFrame에 origin 컬럼 추가
# DILI = DILI.assign(origin='DILI')
# OFT = OFT.assign(origin='OFT')
# T3DB = T3DB.assign(origin='T3DB')
# US = US.assign(origin='US')

# print(f'length of DILI: {len(DILI)}, '
#       f'length of OpenFoodTox: {len(OFT)}, length of T3DB: {len(T3DB)}, '
#       f'length of useful substances: {len(US)}')

In [2]:
# TOXIC_TOTAL = pd.read_csv('data/TOXIC_TOTAL.csv')
# TOXIC_TOTAL

In [3]:
# # TOXIC_TOTAL: 기존 DataFrame, 반드시 'ID', 'Y' 두 컬럼이 존재해야 합니다.
# # 1) 각 규칙에 대한 Boolean 조건 벡터를 만듭니다.
# cond_dili_0 = TOXIC_TOTAL['ID'].str.endswith('_DILI') & (TOXIC_TOTAL['Y'] == 0)
# cond_dili_1 = TOXIC_TOTAL['ID'].str.endswith('_DILI') & (TOXIC_TOTAL['Y'] == 1)
# cond_oft    = TOXIC_TOTAL['ID'].str.endswith('_OFT')
# cond_t3db   = TOXIC_TOTAL['ID'].str.endswith('_T3DB')
# cond_us     = TOXIC_TOTAL['ID'].str.endswith('_US')

# # 2) 조건 순서와 결과 값을 나란히 두고 np.select로 한꺼번에 라벨링
# conditions = [cond_dili_0, cond_dili_1, cond_oft, cond_t3db, cond_us]
# choices    = [0,           1,           0,        1,         0]

# TOXIC_TOTAL['label'] = np.select(conditions, choices, default=np.nan)
# TOXIC_TOTAL['label'].value_counts()
# TOXIC_TOTAL.reset_index(drop=True, inplace=True)
# TOXIC_TOTAL.to_csv('data/TOXIC_TOTAL_labeled.csv', index=False)


# Deepchemberta

In [ ]:
import os
import re
import pickle

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm


base_path = 'data/'

data_COMP_template = os.path.join(base_path, 'SUBDYVE_COMP.csv')
data_SEED_template = os.path.join(base_path, 'SUBDYVE_SEED.csv')



# 사용할 모델 이름 (예: "DeepChem/ChemBERTa-5M-MTR")
model_name = "DeepChem/ChemBERTa-77M-MTR"

# 파일 저장시 모델 경로의 '/' 등을 '_' 로 치환
save_model_prefix = re.sub(r'[/\\]', '_', model_name)

# -----------------------------
# 1. 모델 / 토크나이저 불러오기
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()  # 평가(evaluation) 모드

# -----------------------------
# 2. SMILES → 임베딩 함수
# -----------------------------
def get_embeddings(smiles_list, model, tokenizer, device='cpu'):
    """
    Generate embeddings for a list of SMILES strings.
    Returns a torch.Tensor of shape (len(smiles_list), embedding_dim).
    """
    # 필요시 GPU 사용
    if device == 'cuda' and torch.cuda.is_available():
        model.to(device)
    else:
        device = 'cpu'
        model.to(device)

    # 토큰화
    inputs = tokenizer(
        smiles_list,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512  # SMILES 길이에 따라 적절히 조절
    )

    # to(device)
    for k, v in inputs.items():
        inputs[k] = v.to(device)

    # 모델 추론
    with torch.no_grad():
        outputs = model(**inputs)

    # BERT 류 모델의 마지막 히든 스테이트 (batch_size, seq_len, hidden_dim)
    embeddings = outputs.last_hidden_state.mean(dim=1)  # seq_len에 대해 mean pooling
    # 결과 shape: (batch_size, hidden_dim)

    return embeddings

# -----------------------------
# 3. Seed별로 CSV 읽어서 임베딩 생성 & 저장
# -----------------------------
# for target in tqdm(TARGETS):
print(f"\nCSV 로드 및 임베딩 추출 진행 중...")

# -- (0) data_SEED
data_SEED_path = data_SEED_template
data_SEED_df = pd.read_csv(data_SEED_path)
smiles_SEED = data_SEED_df['SMILES'].tolist()
print(f"len(smiles_SEED): {len(smiles_SEED)}")

embeddings_SEED = get_embeddings(smiles_SEED, model, tokenizer, device='cuda')
# (num_SEED_data, embedding_dim) 형태

parendir = os.path.dirname(data_SEED_path) + '/'
out_fname_SEED = f"{parendir}{save_model_prefix}_TOXIC_Seed.pkl"
with open(out_fname_SEED, "wb") as f:
    pickle.dump(embeddings_SEED.cpu().numpy(), f)
print(f" → data_SEED 임베딩 저장 완료: {out_fname_SEED}")

# -- (1) Data_COMP
data_COMP_path = data_COMP_template
data_COMP_df = pd.read_csv(data_COMP_path)
smiles_COMP = data_COMP_df['SMILES'].tolist()
print(f"len(smiles_COMP): {len(smiles_COMP)}")

embeddings_COMP = get_embeddings(smiles_COMP, model, tokenizer, device='cuda')
# (num_COMP_data, embedding_dim) 형태

parendir = os.path.dirname(data_COMP_path) + '/'
out_fname_COMP = f"{parendir}{save_model_prefix}_TOXIC_Comp.pkl"
with open(out_fname_COMP, "wb") as f:
    pickle.dump(embeddings_COMP.cpu().numpy(), f)
print(f" → data_COMP 임베딩 저장 완료: {out_fname_COMP}")

print("\n모든 Target에 대해 임베딩 추출/저장 완료!")


Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



CSV 로드 및 임베딩 추출 진행 중...
len(smiles_SEED): 256
 → data_SEED 임베딩 저장 완료: data/DeepChem_ChemBERTa-77M-MTR_TOXIC_SEED.pkl
len(smiles_COMP): 317
 → data_COMP 임베딩 저장 완료: data/DeepChem_ChemBERTa-77M-MTR_TOXIC_COMP.pkl

모든 Target에 대해 임베딩 추출/저장 완료!
